# 09 – Price and guest activity
## Question
Do more expensive entire homes accepting short stays have more reviews in the last year?
Keep entire homes with minimum stays of 1–29 nights and positive finite prices; require a nonmissing, finite, nonnegative `number_of_reviews_ltm`. Calculate Spearman's rank correlation separately for each city (two-sided p-values). Repeat only for listings whose first review was at least one calendar year before their own scrape date. Reviews are an activity proxy, not occupancy or revenue.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from src.study_utils import load_spain, export_table, save_figure, SPAIN
from src.cities import CITIES

data = load_spain()
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
from scipy.stats import spearmanr


In [2]:
valid = (data.short_entire & np.isfinite(data.price) & data.price.gt(0)
         & np.isfinite(data.number_of_reviews_ltm) & data.number_of_reviews_ltm.ge(0))
activity = data.loc[valid].copy()
activity['established'] = (activity.first_review <= activity.last_scraped - pd.DateOffset(years=1))
records = []
for city in SPAIN:
    city_data = activity[activity.city.eq(city)]
    for sample, subset in [('All eligible', city_data), ('First review at least one year earlier', city_data[city_data.established])]:
        result = spearmanr(subset.price, subset.number_of_reviews_ltm)
        assert len(subset) > 2 and np.isfinite(result.statistic)
        records.append({'city':city, 'sample':sample, 'n':len(subset), 'rho':result.statistic, 'p_value':result.pvalue})
results = pd.DataFrame(records)
display(export_table(results, 'study09_spearman'))
audit = data.groupby('city').agg(all_listings=('id','size'), short_entire=('short_entire','sum'))
audit['valid_price_and_reviews'] = activity.groupby('city').size()
audit['established'] = activity.groupby('city').established.sum()
display(export_table(audit, 'study09_filter_audit', index=True))


,city,sample,n,rho,p_value
0,barcelona,All eligible,6801,-0.058567,1.343945e-06
1,barcelona,First review at least one year earlier,5185,-0.116452,4.031045e-17
2,madrid,All eligible,12255,-0.011264,2.124384e-01
3,madrid,First review at least one year earlier,8754,-0.004699,6.602542e-01


,all_listings,short_entire,valid_price_and_reviews,established
city,,,,
barcelona,15293,7069,6801,5185
madrid,22708,13903,12255,8754


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, city in zip(axes, SPAIN):
    subset = activity[activity.city.eq(city)]
    ax.scatter(subset.price, subset.number_of_reviews_ltm, s=6, alpha=.18, color='#235789', rasterized=True)
    ax.set_xscale('log')
    row = results[(results.city.eq(city)) & results['sample'].eq('All eligible')].iloc[0]
    ax.set_title(f'{CITIES[city].name}: n={int(row.n):,}, rho={row.rho:.3f}')
    ax.set_xlabel('Quoted nightly price (EUR, log scale)')
axes[0].set_ylabel('Reviews in the last 12 months')
fig.suptitle('Entire homes, minimum stay 1–29 nights; all eligible listings plotted')
fig.tight_layout(); save_figure(fig, 'study09_price_activity'); plt.show()


/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11862/3231446857.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save_figure(fig, 'study09_price_activity'); plt.show()


## Findings and limitations
Compare the sign and magnitude of rho in the full and established samples. The established check reduces exposure differences for recently reviewed listings, but the first review is not a listing creation date and this subset remains selected. Current quoted prices may refer to future stays, while review counts refer to the year before scraping. Unreviewed stays, property quality, capacity, location and pricing responses to demand can all affect the association. P-values do not establish causality and assume more independence than listings managed by shared hosts may provide.


Barcelona has a very weak negative full-sample relationship (rho = −0.059, n = 6,801), becoming −0.116 in the established subset (n = 5,185). Madrid is close to zero in both samples (rho = −0.011 and −0.005; n = 12,255 and 8,754), with p-values above 0.05. These results do not support a positive price–activity association in the observed samples.